In [1]:
# 2025.11.21 경주님 함수 수정 적용 후 
# 99%

import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [2]:
# 데이터 확인하기 2025.11.21 zero_count_rate > 95% 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.preprocessing import StandardScaler # 데이터 전처리용

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing, user_utils

# 모듈 reload
importlib.reload(preprocessing)
importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils    import get_model_train_eval

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 데이터 로딩
train, test = load_data()
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거 

In [4]:
# Data 전처리 1. zero_count_rate이 99%인 컬럼 제거하기 
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.99)


Train Data Analysis (Threshold: 99.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

In [5]:
# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [6]:
# 스케일링
X_train_scaled, X_test_scaled, scaler = scale_data(X_features, X_test)

In [7]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features, 
  y_labels,
)

In [ ]:
# 1. XGBoost, 99%

from xgboost import XGBClassifier

# 모델 이름 생성
model_name = 'XGBoost_99per_basic'

# 변동 없음
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=23
)

# 함수 이용
get_model_train_eval(xgb, model_name, X_train, X_val, y_train, y_val)

# AUC: 0.8417, 정확도: 0.9606, 정밀도: 0.6667, 재현율: 0.0100, F1: 0.0196
# 오차행렬:
# [[14599     3]
#  [  596     6]]
# 실행 시간: 3.200028896331787

✓ 모델 저장 완료: models\XGBoost_99per_basic.pkl
  파일 크기: 1.46 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8417, 정확도: 0.9606, 정밀도: 0.6667, 재현율: 0.0100, F1: 0.0196
오차행렬:
[[14599     3]
 [  596     6]]
실행 시간: 3.200028896331787


In [ ]:
# 2. LogisticRegression 99%

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 모델 이름 생성
model_name = 'LogisticRegression_99per_basic'

log_reg = LogisticRegression(
    max_iter=1000,        # 반복 횟수 충분히 크게
    solver='lbfgs',       # 최적화 알고리즘
    random_state=42,
    n_jobs=-1
)

# 함수 이용
get_model_train_eval(log_reg, model_name, X_train, X_val, y_train, y_val)

# AUC: 0.6229, 정확도: 0.9604, 정밀도: 0.0000, 재현율: 0.0000, F1: 0.0000
# 오차행렬:
# [[14602     0]
#  [  602     0]]
# 실행 시간: 20.425264358520508

✓ 모델 저장 완료: models\LogisticRegression_99per_basic.pkl
  파일 크기: 0.00 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.6229, 정확도: 0.9604, 정밀도: 0.0000, 재현율: 0.0000, F1: 0.0000
오차행렬:
[[14602     0]
 [  602     0]]
실행 시간: 20.425264358520508


c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
# XGBoost 99%
# HyperOpt 최적 하이퍼파라미터 탐색 => 탐색 공간 개선사항 적용
# 규제 파라미터(gamma, min_child_weight) 옵션 까지 포함

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

# 목적 함수 정의
def objective(params):
    model = XGBClassifier(
        n_estimators=int(params['n_estimators']),
        max_depth=int(params['max_depth']),
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        random_state=23,
        n_jobs=-1,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    
    # 학습
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_val)[:, 1]
    
    # 평가 지표 (ROC-AUC 기준)
    auc = roc_auc_score(y_val, y_proba)
    
    return {
        'loss': -auc,   # HyperOpt는 최소화 → 음수로 변환
        'status': STATUS_OK,
        'auc': auc
    }

# 탐색 공간 정의 (개선)
space_xgb = {
    'n_estimators': hp.quniform('n_estimators', 200, 1000, 50),
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'gamma': hp.uniform('gamma', 0, 5),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1)
}
'''
- n_estimators: 너무 크면 과적합, 너무 작으면 학습 부족 → 200~1000 정도 범위
- max_depth: 깊을수록 복잡해짐 → 3~10 범위
- learning_rate: 작은 값(0.01~0.2)으로 두고 n_estimators 크게
- subsample, colsample_bytree: 0.6~1.0 범위로 탐색 (과적합 방지)
'''
# Trials 객체 (결과 저장)
trials = Trials()

# 최적화 실행
best = fmin(
    fn=objective,
    space=space_xgb,
    algo=tpe.suggest,
    max_evals=30,   # 시도 횟수 (필요시 늘리기)
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Best Hyperparameters:", best)

# 100%|██████████| 30/30 [02:13<00:00,  4.45s/trial, best loss: -0.8522921699616992]
# Best Hyperparameters: {
# 'colsample_bytree': np.float64(0.6189145847340869), 
# 'gamma': np.float64(2.1975446881879566), 
# 'learning_rate': np.float64(0.014458073132414333), 
# 'max_depth': np.float64(7.0), 
# 'min_child_weight': np.float64(6.0), 
# 'n_estimators': np.float64(400.0), 
# 'subsample': np.float64(0.8352914228773503)}

  0%|          | 0/30 [00:00<?, ?trial/s, best loss=?]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:37:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



  3%|▎         | 1/30 [00:06<03:03,  6.32s/trial, best loss: -0.7976297221379133]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:37:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



  7%|▋         | 2/30 [00:09<01:58,  4.23s/trial, best loss: -0.8426225347549441]

...

 93%|█████████▎| 28/30 [02:07<00:07,  3.84s/trial, best loss: -0.8522921699616992]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:39:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 97%|█████████▋| 29/30 [02:11<00:03,  3.84s/trial, best loss: -0.8522921699616992]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:39:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



100%|██████████| 30/30 [02:13<00:00,  4.45s/trial, best loss: -0.8522921699616992]
Best Hyperparameters: {'colsample_bytree': np.float64(0.6189145847340869), 'gamma': np.float64(2.1975446881879566), 'learning_rate': np.float64(0.014458073132414333), 'max_depth': np.float64(7.0), 'min_child_weight': np.float64(6.0), 'n_estimators': np.float64(400.0), 'subsample': np.float64(0.8352914228773503)}


In [ ]:
# XGBoost 99%
# 개선된 HyperOpt 적용 

from xgboost import XGBClassifier

# HyperOpt 결과(best) 예시 출력
print("Best Hyperparameters:", best)

# best 딕셔너리에서 값 꺼내기 (개선된 탐색 공간 반영)
# -  gamma와 min_child_weight를 추가
best_params = {
    'n_estimators': int(best['n_estimators']),
    'max_depth': int(best['max_depth']),
    'learning_rate': best['learning_rate'],
    'subsample': best['subsample'],
    'colsample_bytree': best['colsample_bytree'],
    'gamma': best['gamma'],
    'min_child_weight': int(best['min_child_weight'])
}

# 최적 하이퍼파라미터로 모델 생성
model_name = 'XGBoost_99per_hyperopt'

xgb_best = XGBClassifier(
    **best_params,
    random_state=23,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss'
)

# 학습/평가 실행
get_model_train_eval(xgb_best, model_name, X_train, X_val, y_train, y_val)

# AUC: 0.8516, 정확도: 0.9605, 정밀도: 0.6667, 재현율: 0.0066, F1: 0.0132
# 오차행렬:
# [[14600     2]
#  [  598     4]]
# 실행 시간: 2.8659958839416504

Best Hyperparameters: {'colsample_bytree': np.float64(0.6189145847340869), 'gamma': np.float64(2.1975446881879566), 'learning_rate': np.float64(0.014458073132414333), 'max_depth': np.float64(7.0), 'min_child_weight': np.float64(6.0), 'n_estimators': np.float64(400.0), 'subsample': np.float64(0.8352914228773503)}


c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:01:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✓ 모델 저장 완료: models\XGBoost_99per_hyperopt.pkl
  파일 크기: 0.97 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8516, 정확도: 0.9605, 정밀도: 0.6667, 재현율: 0.0066, F1: 0.0132
오차행렬:
[[14600     2]
 [  598     4]]
실행 시간: 2.8659958839416504


In [ ]:
# Logistic Regression 99%
# HyperOpt 최적 하이퍼파라미터 탐색 => 탐색 공간 개선사항 적용
# class_weight 옵션까지 포함

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

# 목적 함수 정의
def objective(params):
    model = LogisticRegression(
        C=params['C'],
        penalty=params['penalty'],
        class_weight=params['class_weight'], # class_weight 추가
        max_iter=1000,
        solver='lbfgs',
        random_state=42,
        n_jobs=-1
    )
    
    # 학습
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_val)[:, 1]
    
    # 평가 지표 (ROC-AUC 기준)
    auc = roc_auc_score(y_val, y_proba)
    
    return {
        'loss': -auc,   # HyperOpt는 최소화 → 음수로 변환
        'status': STATUS_OK,
        'auc': auc
    }

# 탐색 공간 정의
space_log = {
    'C': hp.loguniform('C', np.log(0.001), np.log(10)),
    'penalty': hp.choice('penalty', ['l2']),
    'class_weight': hp.choice('class_weight', [None, 'balanced'])
}
# - gamma, min_child_weight: 규제 파라미터도 탐색 공간에 포함하면 성능 향상 가능

# Trials 객체 (결과 저장)
trials = Trials()

# 최적화 실행
best = fmin(
    fn=objective,
    space=space_log,
    algo=tpe.suggest,
    max_evals=30,   # 시도 횟수
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Best Hyperparameters:", best)

# best 딕셔너리에서 값 꺼내기
best_params = {
    'C': best['C'],
    'penalty': ['l2'][best['penalty']],  # choice 인덱스를 실제 값으로 변환
    'class_weight': [None, 'balanced'][best['class_weight']]
}

# 최적 하이퍼파라미터로 모델 생성
model_name = 'LogisticRegression_99per_hyperopt'

log_reg_best = LogisticRegression(
    **best_params,
    max_iter=1000,
    solver='lbfgs',
    random_state=42,
    n_jobs=-1
)

# 학습/평가 실행
get_model_train_eval(log_reg_best, model_name, X_train, X_val, y_train, y_val)

# 100%|██████████| 30/30 [09:10<00:00, 18.35s/trial, best loss: -0.6423166671292924]
# Best Hyperparameters: {
    # 'C': np.float64(0.02872898758249208), 
    # 'class_weight': np.int64(1), 
    # 'penalty': np.int64(0)
# }

# AUC: 0.6423, 정확도: 0.8954, 정밀도: 0.0589, 재현율: 0.1096, F1: 0.0766
# 오차행렬:
# [[13547  1055]
#  [  536    66]]
# 실행 시간: 16.776742696762085


100%|██████████| 30/30 [09:10<00:00, 18.35s/trial, best loss: -0.6423166671292924]
Best Hyperparameters: {'C': np.float64(0.02872898758249208), 'class_weight': np.int64(1), 'penalty': np.int64(0)}
✓ 모델 저장 완료: models\LogisticRegression_99per_hyperopt.pkl
  파일 크기: 0.00 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.6423, 정확도: 0.8954, 정밀도: 0.0589, 재현율: 0.1096, F1: 0.0766
오차행렬:
[[13547  1055]
 [  536    66]]
실행 시간: 16.776742696762085
